In [11]:
import json
import pandas as pd
import os 

data = []
base_path = os.path.dirname(os.getcwd())
file_path = os.path.join(base_path, "log", "cowrie.json")

with open(file_path) as f:
    data = [json.loads(line) for line in f]
    
df = pd.DataFrame(data)
pd.set_option('display.max_columns', None)
print(df.head(1)) #1 evento -> 1 fila

                  eventid         src_ip  src_port         dst_ip  dst_port  \
0  cowrie.session.connect  101.126.41.73   58982.0  172.31.45.231    2222.0   

        session protocol                                            message  \
0  8fb6f3d375c6      ssh  New connection: 101.126.41.73:58982 (172.31.45...   

             sensor                                  uuid  \
0  ip-172-31-45-231  4a689bfa-1c75-11f1-aaec-e71796b4ff8b   

                     timestamp version hassh hasshAlgorithms kexAlgs keyAlgs  \
0  2026-04-07T07:55:30.885204Z     NaN   NaN             NaN     NaN     NaN   

  encCS macCS compCS langCS duration username password arch input ttylog  \
0   NaN   NaN    NaN    NaN      NaN      NaN      NaN  NaN   NaN    NaN   

   size shasum duplicate outfile destfile  
0   NaN    NaN       NaN     NaN      NaN  


In [28]:
    sessions = []
    grouped = df.groupby("session") #Agrupas el df por sesión
    for session_id, group in grouped:
        #Diccionario de esta sesion
        session_data = {}
    
        #Guarda ID de la sesión
        session_data["session"] = session_id
    
         # IP atacante
        session_data["src_ip"] = group["src_ip"].dropna().iloc[0] if not group["src_ip"].dropna().empty else None
    
        # Puerto origen 
        session_data["src_port"] = group["src_port"].dropna().iloc[0] if not group["src_port"].dropna().empty else None
    
         # Duración segun evento cierre
        duration = group[group["eventid"] == "cowrie.session.closed"]["duration"]
        session_data["duration"] = float(duration.iloc[0]) if not duration.empty else 0
    
        # Login exitoso (1 = si, 0 = no)
        session_data["login_success"] = int((group["eventid"] == "cowrie.login.success").any())
    
        # Intentos de login fallidos
        session_data["login_attempts"] = (group["eventid"] == "cowrie.login.failed").sum()
    
        # Nº comandos ejecutados por el atacante 
        session_data["num_commands"] = (group["eventid"] == "cowrie.command.input").sum()
    
        # Descarga de archivos en la sesión (1 = sí, 0 = no)
        session_data["file_download"] = int((group["eventid"] == "cowrie.session.file_download").any())
    
        #Comandos atacante
        commands = group[group["eventid"] == "cowrie.command.input"]["input"].dropna().tolist()
        session_data["commands"] = commands
    
        #Distintos ataques en base a comandos
        # Reconocimiento
        session_data["recon"] = int(
            any(any(x in cmd.lower() for x in ["uname","whoami","id","ifconfig","pwd"]) for cmd in commands)
        )
    
        # Descarga malware
        session_data["download"] = int(
            any(any(x in cmd.lower() for x in ["wget","curl","tftp"]) for cmd in commands)
        )
    
        # Cambio permisos
        session_data["chmod"] = int(
            any("chmod" in cmd.lower() for cmd in commands)
        )
    
        # Ejecución de binarios/scripts
        session_data["execution"] = int(
            any(any(x in cmd.lower() for x in ["./","sh ","bash "]) for cmd in commands)
        )
    
        # Persistencia
        session_data["persistence"] = int(
            any(".ssh" in cmd.lower() or "authorized_keys" in cmd.lower() for cmd in commands)
        )
        
        # Agrega diccionario de sesión
        sessions.append(session_data)
        
    # Crea dataframe a partir de todos los diccionarios
    dataset = pd.DataFrame(sessions)
    
    print(dataset.head())

        session         src_ip  src_port  duration  login_success  \
0  0562de4fe30a  45.194.37.246   43234.0       4.5              0   
1  0587c101b68a  101.126.41.73   45656.0       2.3              0   
2  0695e64a2061  101.126.41.73   42906.0     120.0              0   
3  08219aebc679  45.194.37.246   47468.0       4.1              0   
4  08ea45147473  45.194.37.246   59112.0       2.3              0   

   login_attempts  num_commands  file_download commands  recon  download  \
0               1             0              0       []      0         0   
1               1             0              0       []      0         0   
2               0             0              0       []      0         0   
3               1             0              0       []      0         0   
4               1             0              0       []      0         0   

   chmod  execution  persistence  
0      0          0            0  
1      0          0            0  
2      0          0    